### 1. streamlit용 모형 만들기
    - y변수명: y
    - 파이프라인

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
import joblib

In [21]:
#모형, X변수 이름을 cols로 저장
#regression/machineCPU.csv

df = pd.read_csv("regression/machineCPU.csv")
X = df.drop(columns=["y"])
y = df["y"]
model1 = Pipeline([
    #("preprocessor", StandardScaler()),
    ("regressor", LinearRegression())
])
model1.fit(X, y)

model2 = RandomForestRegressor(random_state=42)
model2.fit(X, y)

model3 = DecisionTreeRegressor(random_state=42)
model3.fit(X, y)

joblib.dump({"cols": X.columns.tolist(), "model": model1}, "model1.pkl")
joblib.dump({"cols": X.columns.tolist(), "model": model2}, "model2.pkl")
joblib.dump({"cols": X.columns.tolist(), "model": model3}, "model3.pkl")

['model3.pkl']

### 2. streamlit 코드
    - 모형 탑재
    - 입력값 예측
    - .py로 워킹디렉토리에서 streamlit 실행

In [ ]:
import streamlit as st
import pandas as pd
import joblib


st.set_page_config(page_title="회귀모형 예측 실습", layout="centered")

st.title("회귀모형 예측 실습")
st.write("model pkl 파일을 업로드하여 예측")

# pkl 업로드
uploaded_model = st.file_uploader("pkl 모형 업로드", type=["pkl"])

if uploaded_model is not None:
    try:
        loaded = joblib.load(uploaded_model)
        model = loaded["model"]
        st.success("모형 로드 완료")

        # 컬럼 입력
        st.subheader("입력 방식 선택")
        mode = st.radio("선택", ["직접 입력", "CSV 업로드"])

        # 직접 입력
        if mode == "직접 입력":
            st.subheader("직접 입력")

            values = {}
            cols = loaded["cols"]

            for i, col in enumerate(feature_cols):
                with cols[i % len(cols)]:
                    values[col] = st.number_input(f"{col}", value=0.0)

            input_df = pd.DataFrame([values])

            st.write("입력 데이터")
            st.dataframe(input_df, use_container_width=True)

            if st.button("예측하기"):
                pred = model.predict(input_df)[0]
                st.success("예측값: {pred:,.4f}")

        # CSV 업로드
        else:
            st.subheader("CSV 업로드")
            st.write(f"CSV는 다음 변수들을 포함해야 합니다: `{', '.join(loaded['cols'])}`")

            uploaded_csv = st.file_uploader("CSV 파일 업로드", type=["csv"], key="csv")

            if uploaded_csv is not None:
                df = pd.read_csv(uploaded_csv)

                st.write("업로드 데이터")
                st.dataframe(df, use_container_width=True)

                missing_cols = [col for col in loaded['cols'] if col not in df.columns]

                if missing_cols:
                    st.error(f"다음 컬럼이 없습니다: {missing_cols}")
                else:
                    if st.button("일괄 예측"):
                        result_df = df.copy()
                        result_df["prediction"] = model.predict(df[loaded['cols']])

                        st.success("예측 완료")
                        st.dataframe(result_df, use_container_width=True)

                        csv_data = result_df.to_csv(index=False).encode("utf-8-sig")
                        st.download_button(
                            label="결과 CSV 다운로드",
                            data=csv_data,
                            file_name="prediction_result.csv",
                            mime="text/csv"
                        )

    except Exception as e:
        st.error(f"모형 로드 또는 예측 중 오류 발생: {e}")

else:
    st.warning("먼저 pkl 파일을 업로드하세요.")

### 3. API
    - API 값 받아서 예측하도록 API 제공

In [ ]:
# 모델링한 데이터 
#pip install fastapi[standard]
#fastapi dev 파일.py
from fastapi import FastAPI
import pandas as pd
import numpy as np

app = FastAPI()

df = pd.read_csv("regression/machineCPU.csv")
df.drop(columns = "y", inplace=True)

@app.get("/data_num")
def date_gen():
    x = df.sample(1)
    return  x.to_dict(orient="records")


### 4. streamlit으로 API 읽어서 예측, 차트 그리기

In [ ]:
import time
import requests
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st

st.set_page_config(page_title="실시간 예측", layout="wide")
st.title("API 기반 실시간 예측 관리도")


# pkl 업로드
uploaded_model = st.file_uploader("pkl 모형 업로드", type=["pkl"])

if uploaded_model is not None:
    try:
        loaded = joblib.load(uploaded_model)
        model = loaded["model"]
        st.success("모형 로드 완료")
        API_URL = "http://127.0.0.1:8000/data_num"
        feature_cols = loaded["cols"]

        pred_history = []
        step_history = []

        chart_area = st.empty()
        table_area = st.empty()
        status_area = st.empty()

        run = st.button("실행 시작")
        if run:
            for i in range(30):
                try:
                    r = requests.get(API_URL)
                    data = r.json()
                   
                    row_df = pd.DataFrame(data, columns=feature_cols)
                    st.write(f"API 응답 데이터: {data}")
                    time.sleep(3)
                    
                    pred = model.predict(row_df)[0]
                    
                    #Control chart
                    pred_history.append(pred)
                    step_history.append(i + 1)

                    arr = np.array(pred_history)
                    CL = np.mean(arr)
                    sigma = np.std(arr, ddof=1) if len(arr) > 1 else 0
                    UCL = CL + 3 * sigma
                    LCL = CL - 3 * sigma

                    fig, ax = plt.subplots(figsize=(10, 4))
                    ax.plot(step_history, pred_history, marker='o', label="Prediction")
                    ax.axhline(CL, linestyle='--', label=f"CL={CL:.3f}")
                    ax.axhline(UCL, linestyle='--', label=f"UCL={UCL:.3f}")
                    ax.axhline(LCL, linestyle='--', label=f"LCL={LCL:.3f}")
                    ax.set_title("Prediction Control Chart")
                    ax.set_xlabel("Step")
                    ax.set_ylabel("Predicted Value")
                    ax.legend()
                    ax.grid(True)

                    chart_area.pyplot(fig)

                    result_df = pd.DataFrame({
                        "step": step_history,
                        "prediction": pred_history
                    })
                    table_area.dataframe(result_df, use_container_width=True)

                    status_area.success(f"{i+1}번째 데이터 예측 완료: {pred:.4f}")

                except Exception as e:
                    status_area.error(f"오류 발생: {e}")
                    time.sleep(3)

    except Exception as e:
        st.error(f"모형 로드 또는 예측 중 오류 발생: {e}")

else:
    st.warning("pkl 파일 업로드")




